# Re-run video processing

We re-run processing over all files listed in the files_with_proboscis.csv.

We use the _process_video function from choice_assay_pose_processor.py to run two ML models over each video:
- firstly we grab the first frame as an image and run an ML model to find the location of the feeding tubes
- secondly we run an ML model over every frame to identify the location of any bee present (max of 1)


In [ ]:
import platform
from pathlib import Path
from choice_assay.etl.rerun_processing import run_rerun_processing

In [ ]:
# Required config
CONTAINER_NAME = "expidite-choiceassay-trapcam"

# Decide if we're running on Windows or Linux, and set the NAS root path accordingly
running_on_linux = "Linux" in platform.platform()

if running_on_linux:
    nas_root = Path("/bee-ops-disk/")
else:
    nas_root = Path("B://")

    # If we don't have access to the B: drive fall back to a local alternative
    if not nas_root.exists():
        print("Warning: B: drive not found, falling back to local alternative")
        nas_root = Path.home() / "bee-ops-disk"

DOWNLOAD_DIR = nas_root / "azure" / "choice_assay" / "expidite-choiceassay-trapcam"
OUTPUT_DIR = nas_root / "results" / "choice_assay_rerun"

DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Container: {CONTAINER_NAME}")
print(f"Download directory: {DOWNLOAD_DIR.resolve()}")
print(f"Output directory: {OUTPUT_DIR.resolve()}")

In [ ]:
# Create a CSV of all the files in the Azure blobstore container
# This is a one-time operation, so we can comment it out after the first run
from expidite_rpi.core.cloud_connector import CloudConnector
from expidite_rpi.core.configuration import CloudType
import pandas as pd
from pathlib import Path
keys_file = Path.home() / ".expidite" / "keys_choiceassay.env"
cc = CloudConnector.get_instance(CloudType.AZURE)
cc.set_keys(keys_file)

def list_files_in_container(container_name: str, output_csv: Path):
    files = cc.list_cloud_files(container_name)
    df = pd.DataFrame(files, columns=["filename"])
    df.to_csv(output_csv, index=False)

def parse_files_list(csv_file: Path) -> list[str]:
    df = pd.read_csv(csv_file)

    # Drop any row that doesn't have the "_CAVIDEO_" in the filename
    df = df[df["filename"].str.contains("_CAVIDEO_")]

    # Parse out the YYYYMMDD segment from the filename and create a new column for it
    # V3_CAVIDEO_d83add1a11c5_00_00_20260512T015114226_20260512T015154226.mp4
    # Split on the underscores and take the first 8 characters of the 5th element (index 4)
    parts = df["filename"].str.split("_")
    df["device_id"] = parts.str[2]
    df["date"] = parts.str[5].str[:8]

    # Drop any row where date is >20260619
    df = df[df["date"] <= "20260618"]

    # Display a table of the number of files per date and device_id
    print(f"Number of files total: {len(df)}")
    print(df.groupby(["date"]).size())

    # Resave the parsed list to a new CSV
    df.to_csv(csv_file, index=False)

    return df["filename"].tolist()

def download_missing_files(azure_list: Path, local_list: Path):
    df_azure = pd.read_csv(azure_list)
    df_local = pd.read_csv(local_list)

    # The local list includes the directory path, so we need to strip that out to get just the filename
    df_local["filename"] = df_local["filename"].apply(lambda x: Path(x).name)

    # Drop any row that doesn't have the "_CAVIDEO_" in the filename
    df_azure = df_azure[df_azure["filename"].str.contains("_CAVIDEO_")]

    # Get the set of filenames that are in the Azure list but not in the local list
    missing_files = set(df_azure["filename"]) - set(df_local["filename"])
    print(f"Found {len(missing_files)} missing files to download")

    # Download any files that are not already in the download_dir
    cc.download_container(src_container=CONTAINER_NAME,
                          dst_dir=DOWNLOAD_DIR,
                          files=missing_files,
                          overwrite=False)

#list_files_in_container(CONTAINER_NAME, Path("files_list_azure.csv"))
#parse_files_list(Path("files_list_azure.csv"))
#download_missing_files(Path("files_list_azure.csv"), Path("files_list.csv"))

In [ ]:
# Load all the processed_videos_log_ files to get a list of all the processed videos
processed_videos_log_files = list(OUTPUT_DIR.glob("processed_videos_log_*.csv"))
df_list = []
for log_file in processed_videos_log_files:
    print(f"Loading {log_file}")
    df = pd.read_csv(log_file)
    if df.empty:
        print(f"Warning: {log_file} is empty, skipping")
        continue
    df_list.append(df)

all_files = pd.concat(df_list, ignore_index=True) if df_list else pd.DataFrame(columns=["filename"])
if all_files.empty:
    print("Warning: No processed videos found in logs")
else:
    print(f"Total processed videos found in logs: {len(all_files)}")
    all_files.drop_duplicates(keep="last", inplace=True)
print(all_files.columns)
print(all_files["status"].value_counts())
print(f"Total processed videos found: {len(all_files)}")

In [ ]:
# Create a CSV file of all the files in the container.  About 3m for 80k files.
"""
files = DOWNLOAD_DIR.glob(f"{PREFIX}*{SUFFIX}")

# Save the files list to file
files_df = pd.DataFrame([str(f) for f in files], columns=["filename"])
files_df.to_csv("files_list.csv", index=False)
"""

In [ ]:
# Run ML processing over all the video files listed in files_with_proboscis.csv
summary = run_rerun_processing(
    files_to_process=Path("files_list.csv"),
    video_src_dir=DOWNLOAD_DIR,
    output_dir=OUTPUT_DIR,
    file_filter="d83addbca317"
)

print("Run summary:")
print(summary)